In [9]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pdf2image import convert_from_path
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [10]:
print("Loading Model...")
processor = TrOCRProcessor.from_pretrained('openthaigpt/thai-trocr')
model = VisionEncoderDecoderModel.from_pretrained('openthaigpt/thai-trocr')

# Checking GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Loading Model...


Loading weights:   0%|          | 0/490 [00:00<?, ?it/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

# Run

In [68]:
def preprocess_image_for_ocr(pil_img):
    """ล้างภาพให้ขาวดำชัดเจนขึ้น ลด Noise และเส้นตารางจางๆ"""
    # แปลงเป็น OpenCV format
    open_cv_image = np.array(pil_img) 
    # แปลง RGB เป็น BGR
    open_cv_image = open_cv_image[:, :, ::-1].copy() 
    
    # ทำภาพเป็น Grayscale
    gray = cv2.cvtColor(open_cv_image, cv2.COLOR_BGR2GRAY)
    
    # ใช้ Thresholding แบบ OTSU หาจุดตัดขาวดำอัตโนมัติ
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # แปลงกลับเป็น PIL Image เพื่อส่งเข้า TrOCR
    return Image.fromarray(thresh).convert("RGB")

In [69]:
def read_text_from_crop(cropped_img):
    """ส่งภาพที่ตัดแล้วเข้าไปให้ TrOCR อ่าน"""
    # ล้างภาพก่อน
    clean_img = preprocess_image_for_ocr(cropped_img)
    
    pixel_values = processor(images=clean_img, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return generated_text.strip()

In [70]:
ROIs_Config = {
    # 2.1 ข้อมูลสรุป (Summary Section)
    "header": (1800, 100, 2000, 150), # ส่วนหัว (ทดสอบการอ่านเลขไทย)
    "total_voters": (1560, 1100, 1740, 1180),  # จำนวนผู้มีสิทธิเลือกตั้ง (ตัวเลข)
    "turnout_voters": (1560, 1200, 1850, 1300) , # จำนวนผู้มาแสดงตน (ตัวเลข)
    "ballots_allocated": (1350, 1350, 1650, 1450), # จำนวนบัตรเลือกตั้งที่ได้รับจัดสรรมาทั้งหมด
    "ballots_used": (1350, 1350, 1650, 1450)  , # จำนวนบัตรเลือกตั้งที่ใช้ไป    
    "good_ballots": (1200, 1380, 1450, 1460),   # บัตรดี (ตัวเลข)
    "bad_ballots": (1200, 1460, 1450, 1540),    # บัตรเสีย (ตัวเลข)
    "no_vote_ballots": (1200, 1540, 1450, 1620),# บัตรที่ไม่เลือกผู้ใด (ตัวเลข)
    "ballots_remaining": (1350, 1350, 1650, 1450)  , # จำนวนบัตรเลือกตั้งที่เหลือ
    
    # 2.2 ข้อมูลตารางคะแนน (Table Section) - ดึงเฉพาะช่องคะแนนที่เป็นตัวเลข/ตัวอักษร
    "score_candidate_1": (1800, 1900, 2200, 2000), # คะแนนของเบอร์ 1
    "score_candidate_2": (1800, 2000, 2200, 2100), # คะแนนของเบอร์ 2
    "score_candidate_3": (1800, 2100, 2200, 2200), # คะแนนของเบอร์ 3
    "score_candidate_4": (1800, 2200, 2200, 2300), # คะแนนของเบอร์ 4
    "score_candidate_5": (1800, 2300, 2200, 2400)  # คะแนนของเบอร์ 5
}

In [71]:
def process_election_pdf(pdf_path, output_filename):
    print(f"\nProcessing: {pdf_path}")
    
    # --- จัดการโฟลเดอร์ Output ---
    output_dir = "output"
    debug_dir = os.path.join(output_dir, "debug")
    
    # สร้างโฟลเดอร์ถ้ายังไม่มี
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(debug_dir, exist_ok=True)
    # ---------------------------

    print("Converting PDF to Image (300 DPI)...")
    pages = convert_from_path(pdf_path, dpi=300)
    img_page_1 = pages[0]
    
    extracted_data = {}
    
    print("Extracting text via TrOCR...")
    for field_name, box in ROIs_Config.items():
        cropped = img_page_1.crop(box)
        
        # --- เซฟรูป Debug ลงใน /output/debug/ ---
        debug_img_path = os.path.join(debug_dir, f"debug_crop_{field_name}.jpg")
        cropped.save(debug_img_path)
        # ------------------------------------
        
        text = read_text_from_crop(cropped)
        extracted_data[field_name] = text
        print(f" - {field_name}: '{text}'")
        
    df = pd.DataFrame([extracted_data])
    df.insert(0, "source_file", os.path.basename(pdf_path))
    
    # --- เซฟ CSV ลงใน /output/ ---
    csv_file_path = os.path.join(output_dir, output_filename)
    df.to_csv(csv_file_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ Data saved successfully to: {csv_file_path}")
    # -----------------------------
    
    return df

In [72]:
if __name__ == "__main__":
    # ใส่ชื่อไฟล์ PDF ของคุณตรงนี้
    pdf_file_path = "อำเภอบ้านไร่/ตำบลแก่นมะกรูด/หน่วยเลือกตั้งที่ 1/ส.ส.5ทับ18 (บช.).pdf" # หรือ ส.ส.5ทับ18 (บช.).pdf
    output_file = "election_results.csv"
    
    if os.path.exists(pdf_file_path):
        result_df = process_election_pdf(pdf_file_path, output_file)
        print("\nPreview of extracted data:")
        print(result_df)
    else:
        print(f"❌ File not found: {pdf_file_path}. Please check the path.")


Processing: อำเภอบ้านไร่/ตำบลแก่นมะกรูด/หน่วยเลือกตั้งที่ 1/ส.ส.5ทับ18 (บช.).pdf
Converting PDF to Image (300 DPI)...
Extracting text via TrOCR...
 - header: 'ส.ส. ๕/๑๘'
 - total_voters: '828.'
 - turnout_voters: '(__8ใ1... คน (.G'
 - ballots_allocated: 'น...G4__บัตร'
 - ballots_used: 'น...G4__บัตร'
 - good_ballots: 'จํานวน...__'
 - bad_ballots: 'จํานวน...__'
 - no_vote_ballots: 'จํานวน...<?'
 - ballots_remaining: 'น...G4__บัตร'
 - score_candidate_1: 'ตรวจสอบทองพรการการน'
 - score_candidate_2: 'ชและตัวอักษร)ฯ__'
 - score_candidate_3: 'เมโทรทดแทนฝ่ายยุทธการเนรมิตทาง'
 - score_candidate_4: '$ณททนยกทัพหัวเราะ'
 - score_candidate_5: '62 ฯ'

✅ Data saved successfully to: output/election_results.csv

Preview of extracted data:
            source_file     header total_voters    turnout_voters  \
0  ส.ส.5ทับ18 (บช.).pdf  ส.ส. ๕/๑๘         828.  (__8ใ1... คน (.G   

  ballots_allocated  ballots_used good_ballots  bad_ballots no_vote_ballots  \
0      น...G4__บัตร  น...G4__บัตร  จํานวน...__  จํ